# Retail Sales Performance Rollups with PROC SUMMARY

## Executive Summary

A national specialty retailer profiles its **96 stores** — a balanced panel of four regions crossed with three formats — to quantify seasonal revenue, basket traffic, and gross margin. PROC SUMMARY builds analysis-ready rollup datasets at every aggregation level in a single pass, suppressing printed output by default so each step yields a reusable table.

The portfolio averages **$37,546** in seasonal revenue (median $31,318) with a long right tail to **$112,231**. A fully-crossed region x format hierarchy shows revenue tracking store format almost exactly as designed — **Flagship $57,752 vs Standard $37,358 vs Express $17,527** — and every one of the twelve region x format cells follows that same Flagship > Standard > Express order. `IDGROUP` attributes the single top-revenue store in each region (a Flagship store in three of four regions, a Standard store in the Northeast, and all four led by Grocery-dominant stores). Basket-weighted margins rank **Beauty (59.5%) and Apparel (49.8%) highest, Electronics (18.3%) lowest**, and a closing format profile confirms that format drives *volume* — traffic and revenue — while margin is governed by category mix, not format.

## Data Sources

| Dataset | Rows | Grain | Key Variables |
|---------|------|-------|---------------|
| `stores` | 96 | One row per store (a season-to-date sales summary) | `region` (4 levels), `store_format` (3: Flagship/Standard/Express), `category` (the store's dominant department, 5 levels), `store_id`, `revenue` (seasonal USD), `units` (items sold), `baskets` (transaction count), `margin_pct` (gross margin %), `promo_weeks` (promotional weeks run, 1-5) |

The panel is generated inline with `call streaminit` and `rand()`. The portfolio is **balanced by design** — each of the 4 x 3 = 12 region x format combinations holds exactly 8 stores — so region and format are not confounded and every cross-tabulation cell is populated. Revenue scales with format (Flagship 1.8x, Standard 1.0x, Express 0.55x) and the store's dominant category, units track revenue with noise, baskets follow a Poisson count proportional to format scale, and gross margin is category-driven (Beauty and Apparel high, Electronics low). The 96-row size keeps the whole panel within the unlicensed 100-observation cap, so every statistic below is computed on the complete, untruncated data.

# Retail Sales Performance Rollups with PROC SUMMARY

A national specialty retailer needs analysis-ready summaries of its store sales to feed dashboards and merchandising reviews. PROC SUMMARY is the natural tool: it is functionally identical to PROC MEANS but suppresses printed output by default (`NOPRINT`), so it is ideal for producing **output datasets** at multiple aggregation levels in one pass.

In this notebook we:

1. Generate a balanced synthetic store panel (4 regions x 3 formats, 8 stores each).
2. Compute portfolio-wide descriptive statistics with percentiles.
3. Build a fully-crossed (`NWAY`) rollup of revenue and basket metrics.
4. Produce a multi-level hierarchy with `WAYS`/`TYPES` and `CHARTYPE`.
5. Identify the top-revenue store in each region with `IDGROUP`.
6. Report basket-weighted category margins using a `WEIGHT` variable.
7. Profile revenue, traffic, and margin by store format.

## Step 1 — Generate the balanced store panel

Each observation is one store's season-to-date sales summary. We walk the 4 regions x 3 formats grid and place exactly 8 stores in each cell, so the 96-store portfolio is perfectly balanced. Revenue scales with store format (Flagship 1.8x > Standard 1.0x > Express 0.55x) and the store's dominant category; units track revenue with noise, baskets are a Poisson traffic count proportional to format scale, and gross margin is category-driven and eased down slightly by the number of promotional weeks. We fix the seed for reproducibility.

In [1]:
data stores;
    call streaminit(20260531);
    length region $9 store_format $8 category $11;
    array regions[4] $9  _temporary_ ('Northeast' 'Southeast' 'Midwest' 'West');
    array formats[3] $8  _temporary_ ('Flagship' 'Standard' 'Express');
    array cats[5]    $11 _temporary_ ('Apparel' 'Electronics' 'Grocery' 'Home' 'Beauty');

    /* Balanced portfolio: 4 regions x 3 formats x 8 stores = 96 stores */
    do r = 1 to 4;
        region = regions[r];
        do f = 1 to 3;
            store_format = formats[f];

            /* Format-driven revenue scale */
            if      store_format = 'Flagship' then scale = 1.8;
            else if store_format = 'Standard' then scale = 1.0;
            else                                   scale = 0.55;

            do s = 1 to 8;
                store_id = 1000 + (r-1)*24 + (f-1)*8 + s;

                /* Each store has a dominant department */
                cidx     = rand('integer', 1, 5);
                category = cats[cidx];
                if      category = 'Electronics' then do; base = 42; mbase = 18; end;
                else if category = 'Apparel'     then do; base = 28; mbase = 52; end;
                else if category = 'Grocery'     then do; base = 55; mbase = 24; end;
                else if category = 'Home'        then do; base = 24; mbase = 38; end;
                else                                  do; base = 18; mbase = 60; end;

                /* Promotional weeks run this season (of 12) */
                promo_weeks = rand('integer', 1, 5);
                lift        = 1 + 0.18 * (promo_weeks / 12);

                /* Seasonal revenue (USD): format scale x category base x promo lift */
                revenue = round(base * scale * lift
                                * rand('gamma', 12) / 12 * 1000, 0.01);

                /* Units roughly proportional to revenue with noise */
                units   = round(revenue / (12 + 6 * rand('uniform')));

                /* Seasonal basket (transaction) count, scaled by format */
                baskets = rand('poisson', 480 * scale * lift) + 1;

                /* Gross margin %, category dependent; promotions ease it down */
                margin_pct = round(mbase + rand('normal', 0, 3)
                                   - 4 * (promo_weeks / 12), 0.1);
                if margin_pct < 1 then margin_pct = 1;

                output;
            end;
        end;
    end;
    drop r f s scale base mbase lift cidx;
run;

NOTE: DATA stores


NOTE: Wrote stores (96 rows, 9 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds


## Step 2 — Portfolio-wide descriptive statistics

With no CLASS statement, PROC SUMMARY summarizes across all 96 stores. We turn on `PRINT` (since SUMMARY is silent by default) and request a focused set of statistics, including quartiles and the interquartile range. Note the rules of the procedure: only **integer** percentile keywords (`P1`, `P5`, `P10`, `P25`/`Q1`, `MEDIAN`/`P50`, `P75`/`Q3`, `P90`, `P95`, `P99`) are valid here.

In [2]:
proc summary data=stores print maxdec=2
             n mean std min p25 median p75 max qrange;
    var revenue units baskets margin_pct;
run;

                                                  The MEANS Procedure

 Variable          N           Mean     Std Dev     Minimum   Lower Quartile         Median   Upper Quartile        Maximum   Quartile Range
 -------------------------------------------------------------------------------------------------------------------------------------------
 revenue          96       37545.73    25337.11     7013.00         16387.65       31318.19         49809.35      112230.86         33421.70
 units            96        2513.83     1633.01      419.00          1148.75        2190.00          3452.25        7253.00          2303.50
 baskets          96         510.11      193.87      230.00           290.75         509.50           723.50         800.00           432.75
 margin_pct       96          38.67       16.05       11.40            22.55          37.30            52.80          65.40            30.25
 ----------------------------------------------------------------------------------

NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Step 3 — Fully-crossed rollup with NWAY

`NWAY` keeps only the highest `_TYPE_` value, i.e. the full cross of all CLASS variables. This gives one tidy row per region x store_format x category cell — exactly the grain a BI tool wants. Because each store has a single dominant category, the cell `N` is the number of stores falling in that combination. We send the `OUT=` statistics into a named output dataset (no printing) and preview the first 12 cells.

In [3]:
proc summary data=stores nway;
    class region store_format category;
    var revenue units baskets;
    output out=cell_rollup (drop=_type_ _freq_)
        n=stores
        sum=tot_revenue tot_units tot_baskets
        mean=avg_revenue avg_units avg_baskets;
run;

proc print data=cell_rollup (obs=12) noobs label;
    title 'Revenue rollup by Region x Format x Category (first 12 cells)';
run;
title;

                             Revenue rollup by Region x Format x Category (first 12 cells)                              

 region  store_format     category  stores  tot_revenue  tot_units  tot_baskets       avg_revenue        avg_units     avg_baskets
Midwest  Express       Apparel           1      9778.28        577          296           9778.28              577             296
Midwest  Express       Beauty            2     23073.96       1719          527          11536.98            859.5           263.5
Midwest  Express       Electronics       1     28365.63       1731          274          28365.63             1731             274
Midwest  Express       Grocery           1     26359.03       1609          263          26359.03             1609             263
Midwest  Express       Home              3     47840.33       3157          802  15946.7766666667  1052.3333333333  267.3333333333
Midwest  Flagship      Apparel           1     60968.53       4466          724          609

NOTE: PROC MEANS
NOTE: Output dataset cell_rollup has 53 observations and 10 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC PRINT data=cell_rollup

NOTE: PROC PRINT completed: 12 observations printed, 10 variables


## Step 4 — Multi-level hierarchy with WAYS, TYPES and CHARTYPE

Instead of only the full cross, a merchandising report often needs several aggregation levels at once: the grand total, each single dimension, and the region x format combination. `WAYS 0 1 2` requests aggregations of those sizes; `CHARTYPE` writes `_TYPE_` as a readable character bitmask; `COMPLETETYPES` and `PRINTALLTYPES` print every level. Because the panel is balanced (8 stores per region x format cell), all twelve two-way cells are populated. We restrict the analysis variable to revenue here for clarity.

In [4]:
proc summary data=stores chartype completetypes
             printalltypes print maxdec=0
             sum mean;
    class region store_format;
    var revenue;
    ways 0 1 2;
run;

                                                  The MEANS Procedure

                                              Analysis Variable : revenue

                    N
                  Obs            Sum           Mean
        -------------------------------------------
                   96        3604390          37546
        -------------------------------------------

                                              Analysis Variable : revenue

                                  N
        store_format            Obs            Sum           Mean
        ---------------------------------------------------------
        Express                  32         560851          17527
        Flagship                 32        1848077          57752
        Standard                 32        1195462          37358
        ---------------------------------------------------------

                                              Analysis Variable : revenue

                               N
       

NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Step 5 — Top-revenue store in each region (IDGROUP)

District managers want to know the single best-performing store in each region — and *which* store it is. Because the panel is already one row per store, we summarize directly by `region` and use `IDGROUP(MAX(revenue) OUT(...)=...)` to carry the identifying `store_id`, `store_format`, and dominant `category` of the maximum-revenue store into the regional summary. We also report `MEAN` so the leader can be read against its regional average — the gap that a flat `MAX=` alone cannot attribute.

In [5]:
/* The panel is already one row per store, so summarize straight by region.
   IDGROUP(MAX(...)) carries the winning store's identifying attributes. */
proc summary data=stores nway;
    class region;
    var revenue;
    output out=top_store (drop=_type_ _freq_)
        max=best_store_revenue
        mean=region_avg_revenue
        idgroup(max(revenue)
                out(store_id store_format category)=
                    best_store_id best_store_format best_category);
run;

proc print data=top_store noobs label;
    title 'Top-revenue store in each region';
run;
title;

                                            Top-revenue store in each region                                            

   region  best_store_revenue  region_avg_revenue  best_store_id  best_store_format  best_category
Midwest             112230.86    41131.3929166667           1055  Flagship           Grocery
Northeast            80846.33    35742.1854166667           1012  Standard           Grocery
Southeast           111961.62    36542.1283333333           1029  Flagship           Grocery
West                  92619.6    36767.1995833333           1073  Flagship           Grocery



NOTE: PROC MEANS
NOTE: Output dataset top_store has 4 observations and 6 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC PRINT data=top_store

NOTE: PROC PRINT completed: 4 observations printed, 6 variables


## Step 6 — Weighted category margins

A plain average margin across stores treats a low-traffic Express store the same as a busy Flagship. Weighting each store's `margin_pct` by its basket count yields a customer-weighted margin that reflects where transactions actually happen. The `WEIGHT` statement applies the weight to the statistics, and `VARDEF=WEIGHT` uses the sum of weights as the variance divisor. We then sort the categories high-to-low.

In [6]:
proc summary data=stores nway vardef=weight;
    class category;
    weight baskets;
    var margin_pct;
    output out=cat_margin (drop=_type_ _freq_)
        mean=wtd_margin_pct
        std=wtd_margin_std
        sumwgt=total_baskets;
run;

proc sort data=cat_margin; by descending wtd_margin_pct; run;

proc print data=cat_margin noobs label;
    var category wtd_margin_pct wtd_margin_std total_baskets;
    format wtd_margin_pct wtd_margin_std 6.2 total_baskets comma12.;
    title 'Basket-weighted gross margin by category (high to low)';
run;
title;

                                 Basket-weighted gross margin by category (high to low)                                 

   category  wtd_margin_pct  wtd_margin_std  total_baskets
Beauty                59.54            3.11         11,337
Apparel               49.78            2.54          9,113
Home                  36.41            2.68         11,392
Grocery               22.96            2.19          8,213
Electronics           18.32            3.12          8,916



NOTE: PROC MEANS
NOTE: Output dataset cat_margin has 5 observations and 4 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC SORT data=cat_margin

NOTE: Read 5 rows from cat_margin.
NOTE: Wrote cat_margin (5 rows, 4 columns).
NOTE: PROC SORT statement used.
NOTE: PROC PRINT data=cat_margin

NOTE: PROC PRINT completed: 5 observations printed, 4 variables


## Step 7 — Revenue, traffic and margin profile by store format

Finally, we profile the three store formats side by side to confirm the design intent and surface a managerial insight: format should drive revenue and traffic (Flagship was built with a 1.8x scale, Express 0.55x), but gross margin — being category-driven — should be roughly flat across formats. `CLASS ... / ORDER=DATA` keeps the formats in their natural Flagship/Standard/Express order, and we emit a clean comparison dataset alongside the printed table.

In [7]:
proc summary data=stores nway;
    class store_format / order=data;
    var revenue units baskets margin_pct;
    output out=format_profile (drop=_type_ _freq_)
        n=n_stores
        mean=avg_revenue avg_units avg_baskets avg_margin_pct;
run;

proc print data=format_profile noobs label;
    format avg_revenue avg_units avg_baskets comma10.
           avg_margin_pct 6.2;
    title 'Revenue, traffic and margin profile by store format';
run;
title;

                                  Revenue, traffic and margin profile by store format                                   

store_format  n_stores  avg_revenue  avg_units  avg_baskets  avg_margin_pct
Express             32       17,527      1,213          276           39.83
Flagship            32       57,752      3,794          743           39.35
Standard            32       37,358      2,535          511           36.83



NOTE: PROC MEANS
NOTE: Output dataset format_profile has 3 observations and 6 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC PRINT data=format_profile

NOTE: PROC PRINT completed: 3 observations printed, 6 variables


## Interpreting the results

- **Portfolio profile (Step 2).** Across all 96 stores, seasonal revenue averages **$37,546** with a median of **$31,318** and an interquartile range (`QRANGE`) of **$33,422**; the gap between the upper quartile (~$49,809) and the maximum (**$112,231**) flags a long right tail of high-revenue Flagship stores. Margin centers near **38.7%** but spans 11–65%, reflecting the wide category mix.
- **Cell rollup (Step 3).** The `NWAY` dataset is dashboard-ready at the finest grain (53 populated region x format x category cells out of 60 possible). Within the Flagship tier the high-base categories — Grocery and Electronics — post the largest per-store revenue (e.g. a Midwest Flagship Grocery store averages ~$107,261), consistent with the design.
- **Hierarchy (Step 4).** `WAYS 0 1 2` returns the grand total ($3.6M, mean $37,546), each single dimension, and the region x format combination in one table. Revenue tracks format almost exactly as built: **Flagship $57,752 > Standard $37,358 > Express $17,527** (a 3.3x Flagship-to-Express ratio, matching the 1.8 / 0.55 scale ratio). Regions are close (Midwest $41,131 highest, Northeast $35,742 lowest), and because the panel is balanced, **all twelve region x format cells are populated** — every one preserving the Flagship > Standard > Express order.
- **Top stores (Step 5).** `IDGROUP(MAX(...))` returns not just the best revenue per region but the specific store achieving it. Each regional leader earns **two to three times its regional average** (e.g. Midwest store **1055** at **$112,231** vs a $41,131 average). Three of the four leaders are **Flagship** stores; the Northeast's top store (**1012**) is a **Standard** store, and notably **all four leaders are Grocery-dominant** — an attribution a flat `MAX=` could never provide.
- **Weighted margin (Step 6).** Basket-weighting orders categories by structural margin: **Beauty 59.5% > Apparel 49.8% > Home 36.4% > Grocery 23.0% > Electronics 18.3%**. High-margin Beauty and Apparel lead while Electronics trails, reflecting its structurally thin margin; weighting by baskets (Home and Beauty each ~11,300 baskets) keeps busy stores from being drowned out by sparse ones.
- **Format profile (Step 7).** Flagship stores average **$57,752** revenue, **3,794** units, and **743** baskets — roughly three times an Express store's **$17,527 / 1,213 / 276**. Yet average margin is **nearly flat across formats** (Flagship 39.4%, Express 39.8%, Standard 36.8%): format drives *volume*, while margin is governed by category mix. That separation is exactly what merchandising and real-estate teams need to plan independently.

Because PROC SUMMARY emits datasets by default, every step above produces a reusable table (`cell_rollup`, `top_store`, `cat_margin`, `format_profile`) that can feed downstream reporting or visualization without additional data wrangling.